In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
import json
import time
import os
import pickle
import multiprocessing as mp
from importlib import reload
import pydantic
import yaml

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Add the parent directory to the system path to import modules
# sys.path.append(str(Path(__file__).absolute().parent))
if 'haman' in os.getcwd():
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester')
    sys.path.append('/home/haman/mf-temporary/MeanFieldTester/codes')
    os.chdir('/home/haman/mf-temporary/MeanFieldTester')
    project_path = Path('/home/haman/mf-temporary')
elif 'pavel' in os.getcwd():
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester')
    sys.path.append('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester/codes')
    os.chdir('/home/pavel/academia/wintermute/mf-temporary/MeanFieldTester')
    project_path = Path('/home/pavel/academia/wintermute/mf-temporary')

import codes.controller as rw
import codes.controller.config as cfg
from codes.plotting import fig_plots
from codes import plotting as ax_plt
import codes.cell_library as cells
import codes.neuron_simulation as ns
import codes.meanfield_simulation as mfs
import codes.network_simulation as nets
import codes.transfer_function as tf
from codes import utils

from codes.tvb_models.models import Zerlaut_adaptation_first_order


In [ ]:
# Zerlaut network params to have a reference for testing

network_params = {
    "exc_neuron" : {                            # taken from diVolo2019_BiologicallyRealistic
        "neuron_params" : {
            "v_rest" : -65.0,                   # [mV], Resting membrane potential
            "v_reset" : -65.0,                  # [mV], Reset potential after spike
            "tau_refrac" : 5.0,                 # [ms], Refractory period
            "tau_m" : 15.0,                     # [ms], Membrane time constant, taken as cm/gl
            "cm" : 0.150,                       # [nF], Membrane capacitance
            "e_rev_E" : 0.0,                    # [mV], Excitatory reversal potential
            "e_rev_I" : -80.0,                  # [mV], Inhibitory reversal potential
            "tau_syn_E" : 5.0,                  # [ms], Excitatory synaptic time constant
            "tau_syn_I" : 5.0,                  # [ms], Inhibitory synaptic time constant
            "a" : 4.0,                          # [nS], Subthreshold adaptation conductance
            "b" : 0.02,                         # [nA], Spike-triggered adaptation increment
            "delta_T" : 2.0,                    # [mV], Slope factor
            "tau_w" : 500.0,                    # [ms], Adaptation time constant
            "v_thresh" : -50.0                  # [mV], Spike threshold
        },
        "init_values" : {
            "w" : 0.00,                         # [nA], Default value of adaptation current
            "v" : -65.0,                        # [mV], Default value of membrane potential
        },    
    },
    "inh_neuron" : {                            # taken from diVolo2019_BiologicallyRealistic
        "neuron_params" : {
            "v_rest" : -65.0,                   # [mV], Resting membrane potential
            "v_reset" : -65.0,                  # [mV], Reset potential after spike
            "tau_refrac" : 5.0,                 # [ms], Refractory period
            "tau_m" : 15.0,                     # [ms], Membrane time constant, taken as cm/gl
            "cm" : 0.150,                       # [nF], Membrane capacitance
            "e_rev_E" : 0.0,                    # [mV], Excitatory reversal potential
            "e_rev_I" : -80.0,                  # [mV], Inhibitory reversal potential
            "tau_syn_E" : 5.0,                  # [ms], Excitatory synaptic time constant
            "tau_syn_I" : 5.0,                  # [ms], Inhibitory synaptic time constant
            "a" : 0.0,                          # [nS], Subthreshold adaptation conductance
            "b" : 0.0,                          # [nA], Spike-triggered adaptation increment
            "delta_T" : 0.5,                    # [mV], Slope factor
            "tau_w" : 500.0,                    # [ms], Adaptation time constant
            "v_thresh" : -50.0                  # [mV], Spike threshold
        },
        "init_values" : {
            "w" : 0.00,                         # [nA], Default value of adaptation current
            "v" : -65.0,                        # [mV], Default value of membrane potential
        },    
    },
    "network" : {
        "total_pop_size" : 10000,                  # int, Total number of exc and inh neurons
        "drive_pop_size" : 8000,                     # int, number of driving neurons
        "stim_pop_size" : 8000,
        "g" : 0.2,                                  # float in [0,1], percentage of inhibitory neurons
        "p_connect_exc" : 0.05,                       # Probability of incoming connections coming from excitatory neurons
        "p_connect_inh" : 0.05,                       # Probability of incoming connections coming from inhibitory neurons
        "p_connect_drive" : 0.05,
        "p_connect_stim" : 0.05
    },
    "synapses" : {
        "exc_synapses" : {
            "syn_params" : {
                "weight" : 1.0,                     # [nS], synaptic weight
                "delay" : 1.0,                      # [ms], synaptic delay
            },
            "syn_type" : "static_synapse",          # str, nest synapse type
        },
        "inh_synapses" : {
            "syn_params" : {
                "weight" : 5.0,                     # [nS], synaptic weight
                "delay" : 1.0,                      # [ms], synaptic delay
            },
            "syn_type" : "static_synapse",          # str, nest synapse type
        },
        "drive_synapses" : {
            "syn_params" : {
                "weight" : 1.0,                     # [nS], synaptic weight
                "delay" : 1.0,                      # [ms], synaptic delay
            },
            "syn_type" : "static_synapse",         # str, nest synapse type
        },
        "stim_synapses" : {
            "syn_params" : {
                "weight" : 1.0,                     # [nS], synaptic weight
                "delay" : 1.0,                      # [ms], synaptic delay
            },
            "syn_type" : "static_synapse",        # str, nest synapse type
        },
    },
    "transfer_function" : {
        "expansion_point" : [-60, 4, 0.5],
        "expansion_norm" : [10, 6, 1.0],
        "exc_neuron" : [-49.8, 5.06, -25.0, 1.4, -0.41, 10.5, -36.0, 7.4, 1.2, -40.7],
        "inh_neuron" : [-51.4, 4.0, -8.3, 0.2, -0.5, 1.4, -14.6, 4.5, 2.8, -15.3],
    }
}

In [ ]:
project_path = Path.cwd().parent / "projects" / "00_drafts_tests"
cells.update_params(network_params)
neuron_names = ["exc_neuron", "inh_neuron"]
neuron_params = {neuron: network_params[neuron] for neuron in neuron_names}


# Testing of `neuron_simulation` module

I have made big changes in neuron_simulation
- I have added new way of handling config parameters (using pydantic)
- I have split `__init__.py` into three files
    - `__init__.py` - importing and handling backend simulators and providing `run_neuron_simulation_workflow` function 
    - `__base__.py` - defines an abstract class of simulator, which should be the main implementation of each simulator backend
    - `pynn_simulator.py` previous simulation code with extra improvements

### Testing linear grid

In [ ]:
test_workflow_params = {
    "neuron_simulation": {
        "execution_mode": "run",
        "simulator": "pynn.nest",
        "grid": {
            "exc_neuron" : {
                "grid_type": "linear",
                "exc_rate_grid": [0, 30, 16],  # Quick small grid for testing
                "inh_rate_grid": [0, 30, 16],
            },
            "inh_neuron" : {
                "grid_type": "linear",
                "exc_rate_grid": [0, 30, 16],  # Quick small grid for testing
                "inh_rate_grid": [0, 30, 16]
            }
        },
        "simulation_time": 5000.0,
        "averaging_window": 2000.0,
        "time_step": 0.1,
        "seed": 42,
        "n_runs": 5,
        "cpus": 16
    }
}

test_config = cfg.WorkflowConfig(**test_workflow_params)

print("Config loaded successfully!")
print(f"Running for {test_config.neuron_simulation.simulation_time} ms")

neuron_results = ns.run_neuron_simulation_workflow(test_config.neuron_simulation, neuron_params)

In [ ]:
fig_name = "neuron_activity_test_linear_grid.png"
fig_path = project_path / fig_name
fig_plots.fig_neuron_activity(
    neuron_results, 
    common_params={}, 
    fig_params={
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
    }
)
display(Image(filename=str(fig_path)))

fig_name = "std_neuron_activity_test_linear_grid.png"
fig_path = project_path / fig_name
fig_plots.fig_tf_fits_together(
    neuron_results, 
    {"exc_neuron": [],
     "inh_neuron": []
     }, 
    common_params={
        'labels' : [],
        'linestyles' : [],
        # 'xlim' : (0,7),
        'ylim' : (0, 60),
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
    })
display(Image(filename=str(fig_path)))

### Testing adaptive grid

In [ ]:
test_workflow_params = {
    "neuron_simulation": {
        "execution_mode": "run",
        "simulator": "pynn.nest",
        "grid": {
            "exc_neuron" : {
                "grid_type": "adaptive",
                "exc_rate_grid": "adaptive",
                "inh_rate_grid": [0, 30, 16],
                "out_rate_grid": [0, 30, 16],
                "n_coarse_interpolation_points" : 16, 
            },
            "inh_neuron" : {
                "grid_type": "adaptive",
                "exc_rate_grid": "adaptive",
                "inh_rate_grid": [0, 30, 16],
                "out_rate_grid": [0, 30, 16],
                "n_coarse_interpolation_points" : 16, 
            }
        },
        "simulation_time": 5000.0,
        "averaging_window": 2000.0,
        "time_step": 0.1,
        "seed": 42,
        "n_runs": 5,
        "cpus": 16
    }
}

test_config = cfg.WorkflowConfig(**test_workflow_params)

print("Config loaded successfully!")
print(f"Running for {test_config.neuron_simulation.simulation_time} ms")

neuron_results = ns.run_neuron_simulation_workflow(test_config.neuron_simulation, neuron_params)

In [ ]:
fig_name = "neuron_activity_test_adaptive_grid.png"
fig_path = project_path / fig_name
fig_plots.fig_neuron_activity(
    neuron_results, 
    common_params={}, 
    fig_params={
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
    }
)
display(Image(filename=str(fig_path)))

fig_name = "std_neuron_activity_test_adaptive_grid.png"
fig_path = project_path / fig_name
fig_plots.fig_tf_fits_together(
    neuron_results, 
    {"exc_neuron": [],
     "inh_neuron": []
     }, 
    common_params={
        'labels' : [],
        'linestyles' : [],
        # 'xlim' : (0,7),
        'ylim' : (0, 30),
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
    })
display(Image(filename=str(fig_path)))

### Testing custom grid

In [ ]:
data_path = project_path.parent / "01_fitting_tf" / "zerlaut2018_data"
custom_grid = {
    "exc_neuron": {},
    "inh_neuron": {}
}

for neuron_name, cell_type in zip(["exc_neuron", "inh_neuron"], ["RS", "FS"]):
    data = np.load(data_path / f"{cell_type}-cell_CONFIG1.npy", allow_pickle=True)
    custom_grid[neuron_name] = {
        "grid_type": "custom",
        "exc_rate_grid": data[2].T,
        "inh_rate_grid": np.stack([data[3]]*10, axis=1).T,
    }


In [ ]:
test_workflow_params = {
    "neuron_simulation": {
        "execution_mode": "run",
        "simulator": "pynn.nest",
        "grid": custom_grid,
        "simulation_time": 5000.0,
        "averaging_window": 2000.0,
        "time_step": 0.1,
        "seed": 42,
        "n_runs": 5,
        "cpus": 16
    }
}

test_config = cfg.WorkflowConfig(**test_workflow_params)

print("Config loaded successfully!")
print(f"Running for {test_config.neuron_simulation.simulation_time} ms")

neuron_results = ns.run_neuron_simulation_workflow(test_config.neuron_simulation, neuron_params)

In [ ]:
fig_name = "neuron_activity_test_custom_grid.png"
fig_path = project_path / fig_name
fig_plots.fig_neuron_activity(
    neuron_results, 
    common_params={}, 
    fig_params={
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
    }
)
display(Image(filename=str(fig_path)))

fig_name = "std_neuron_activity_test_custom_grid.png"
fig_path = project_path / fig_name
fig_plots.fig_tf_fits_together(
    neuron_results, 
    {"exc_neuron": [],
     "inh_neuron": []
     }, 
    common_params={
        'labels' : [],
        'linestyles' : [],
        # 'xlim' : (0,7),
        'ylim' : (0, 30),
    }, 
    fig_params={
        'fontsize': 14,
        'figsize': (15, 10),  # width, height
        'tight_layout': True,
        'savefig': True,
        'savefig_path': fig_path,
    })
display(Image(filename=str(fig_path)))

### Testing cpu=1

### Other tests
- `execution_mode = "load"`
- test if it fails properly, (test the inputs such, give wrong numbers, values etc.)
- test optional values
- test seeds (two runs with the same seeds if the results are equal)